# Sanity load DiT-XL/2 — Phase 0 Task 0.3

Load `facebook/DiT-XL-2-256` via `diffusers.DiTPipeline`, measure VRAM in fp32 and fp16, generate one class-conditional sample (ImageNet class 207 = golden retriever), time the sampling wall-clock.

No attacks, no metrics, no backward pass — those come in Phase 1+. This notebook is the observation log of a single successful run; the compute is wrapped in [`scripts/sanity_load_dit.py`](../scripts/sanity_load_dit.py) for re-execution.

**Environment**: AMD RX 6900 XT (16 GB), ROCm 7.2.2, PyTorch 2.11.0+rocm7.2, diffusers 0.37.1, Python 3.12 (uv).

**Constraint carried over from PROJECT_PLAN §3**: bf16 is ~5× slower than fp32 on RDNA 2 — only fp16 or fp32 here, never bf16.

In [1]:
import os
import time
from pathlib import Path

REPO_ROOT = Path.cwd().parent
os.environ.setdefault("HF_HOME", str(REPO_ROOT / ".hf_cache"))

import torch
from diffusers import DiTPipeline

MODEL_ID = "facebook/DiT-XL-2-256"
OUT_DIR = REPO_ROOT / "notebooks" / "out"
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available(), "CUDA/ROCm not available"
print(f"[env] torch {torch.__version__} | device: {torch.cuda.get_device_name(0)}")
print(f"[env] HF_HOME = {os.environ['HF_HOME']}")
print(f"[env] model: {MODEL_ID}")

def vram_gb() -> float:
    return torch.cuda.memory_allocated() / (1024 ** 3)

[env] torch 2.11.0+rocm7.2 | device: AMD Radeon RX 6900 XT
[env] HF_HOME = /home/marco/disattend/.hf_cache
[env] model: facebook/DiT-XL-2-256


## fp32 load + VRAM

Baseline footprint — the raw weight size on device. Expected ~2.6 GB from param count alone; any extra is VAE + class embedder + overhead.

In [2]:
print("[fp32] loading DiT-XL/2 in fp32 ...")
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
t0 = time.perf_counter()
pipe32 = DiTPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.float32).to("cuda")
torch.cuda.synchronize()
load_fp32_sec = time.perf_counter() - t0
vram_fp32 = vram_gb()
print(f"[fp32] load wallclock: {load_fp32_sec:.2f} s | VRAM allocated: {vram_fp32:.3f} GB")

del pipe32
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

[fp32] loading DiT-XL/2 in fp32 ...
[fp32] load wallclock: 15.33 s | VRAM allocated: 3.134 GB


## fp16 load + VRAM

Load the pipeline fresh in fp16 (not an in-place cast of the fp32 model — cleaner memory measurement).

In [3]:
print("[fp16] loading DiT-XL/2 in fp16 ...")
t0 = time.perf_counter()
pipe = DiTPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.float16).to("cuda")
torch.cuda.synchronize()
load_fp16_sec = time.perf_counter() - t0
vram_fp16 = vram_gb()
scheduler_name = pipe.scheduler.__class__.__name__
print(f"[fp16] load wallclock: {load_fp16_sec:.2f} s | VRAM allocated: {vram_fp16:.3f} GB")
print(f"[fp16] default scheduler: {scheduler_name}")

[fp16] loading DiT-XL/2 in fp16 ...
[fp16] load wallclock: 0.97 s | VRAM allocated: 1.583 GB
[fp16] default scheduler: DDIMScheduler


## Sample — class 207 (golden retriever), 250 steps

The default scheduler turns out to be **DDIM** (not DDPM as the plan guessed). That's a lucky find: the Phase 2 attack wants deterministic, differentiable sampling exactly for this reason — see `docs/phase0_plan.md` §5 open decision "scheduler per attacco futuro".

In [4]:
CLASS_LABEL = 207  # ImageNet class 207 = golden retriever
NUM_INFERENCE_STEPS = 250
SEED = 42

print(f"[sample] class {CLASS_LABEL} (golden retriever), {NUM_INFERENCE_STEPS} {scheduler_name} steps, seed {SEED}")
generator = torch.Generator(device="cuda").manual_seed(SEED)
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()
t0 = time.perf_counter()
with torch.no_grad():
    result = pipe(
        class_labels=[CLASS_LABEL],
        num_inference_steps=NUM_INFERENCE_STEPS,
        generator=generator,
    )
torch.cuda.synchronize()
sample_sec = time.perf_counter() - t0
vram_peak_sample = torch.cuda.max_memory_allocated() / (1024 ** 3)
print(f"[sample] wallclock: {sample_sec:.2f} s | VRAM peak during sampling: {vram_peak_sample:.3f} GB")

img = result.images[0]
png_path = OUT_DIR / "golden_retriever_fp16.png"
img.save(png_path)
print(f"[sample] saved: {png_path.relative_to(REPO_ROOT)}")

[sample] class 207 (golden retriever), 250 DDIMScheduler steps, seed 42
[sample] wallclock: 27.99 s | VRAM peak during sampling: 1.982 GB
[sample] saved: notebooks/out/golden_retriever_fp16.png


## Result

![golden retriever 256×256, DiT-XL/2 fp16, 250 DDIM steps, seed 42](out/golden_retriever_fp16.png)

| metric                     | fp32      | fp16      |
|----------------------------|-----------|-----------|
| VRAM after load            | 3.134 GB  | 1.583 GB  |
| Load wall-clock            | 15.33 s   | 0.97 s    |
| Sampling 250 steps         | —         | 27.99 s   |
| VRAM peak during sampling  | —         | 1.982 GB  |

fp16 is comfortably within the 16 GB VRAM budget (~12% of VRAM at peak). Headroom for the batched PGD forward/backward needed in Phase 2 is plentiful, even without gradient checkpointing.

The fp32 footprint is ~0.5 GB above the 2.6 GB weights-only estimate — the delta is VAE (`AutoencoderKL`, ~335 MB fp32) + class embedder + small tensors. Consistent with expectations.

Sampling rate fp16: ~18 it/s steady state, i.e. ~55 ms per DiT forward on 256×256 latents. For a 50-step DDIM (the probable attack schedule), wall-clock would be ~3 s per image — acceptable for the inner loop of a ~20-step PGD attack.

## Observations worth carrying into Phase 1

- **Default scheduler is `DDIMScheduler`**, not DDPM. `docs/phase0_plan.md` §5 can close the "scheduler per attacco futuro" decision towards DDIM with high confidence — it's already the pipeline default.
- **`safetensors` not on this checkpoint**: HF logs `Error no file named diffusion_pytorch_model.safetensors` and falls back to pickled `.bin`. Benign, but worth noting for reproducibility audit (and for the `safetensors` dep — it's needed by `diffusers` in general, not by this specific checkpoint).
- **`accelerate` missing warning** is a soft recommendation for faster loading (`low_cpu_mem_usage`). Not installing it per PROJECT_PLAN §3 — load time is already <1 s cold-cache fp16.
- **fp16 VRAM 1.58 GB ≠ half of fp32 3.13 GB**: the 0.33 GB gap is the `AutoencoderKL` VAE + allocator overhead. The transformer itself (the part that matters for the attack) does halve cleanly.
